# LiveCam AgriVision - training on Colab (GPU)

Runtime -> Change runtime type -> **T4 GPU**. Put your prepared datasets (or raw data + mapping files)
in Google Drive under `MyDrive/agrivision/`.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/agrivision'

In [ ]:
!git clone https://github.com/jyothirmaisadhu/livecam-agrivision.git
%cd livecam-agrivision
!pip install -q -r requirements.txt && pip install -q -e .

## Step 1 - datasets
Either copy already-prepared `data/` from Drive, or build it from raw sources with your mapping files.

In [ ]:
import os
if os.path.exists(f'{DRIVE}/data'):
    !cp -r {DRIVE}/data ./data
else:
    !python -m agrivision.data.prepare_classification --mapping {DRIVE}/disease_sources.yaml --source-root {DRIVE}/raw --out data/disease
    !python -m agrivision.data.prepare_yolo --mapping {DRIVE}/pest_sources.yaml --source-root {DRIVE}/raw --out data/pest
!ls data

## Step 2 - MobileNetV3 disease classifier

In [ ]:
!python -m agrivision.disease.train --data data/disease --out models/disease --variant large --epochs-head 10 --epochs-finetune 15 --batch 32
!python -m agrivision.disease.evaluate --data data/disease --model-dir models/disease
from IPython.display import Image
Image('models/disease/confusion_matrix.png')

## Step 3 - YOLO11n pest detector

In [ ]:
!python -m agrivision.pest.train --data data/pest/data.yaml --out models/pest --epochs 100 --batch 16 --imgsz 640 --device 0

## Step 4 - U-Net damage segmentation (needs data/damage/images + masks)

In [ ]:
!python -m agrivision.damage.train --data data/damage --out models/damage --epochs 40 --batch 8

## Save models to Drive and try the pipeline on a video

In [ ]:
!mkdir -p {DRIVE}/models && cp -r models/* {DRIVE}/models/
# !python -m agrivision.pipeline.live --source {DRIVE}/field_video.mp4 --no-display --save runs/live.mp4 \
#     --disease models/disease/disease_mobilenetv3.tflite --labels models/disease/labels.json \
#     --pest models/pest/pest_yolo11n.pt --damage models/damage/damage_unet.tflite